# Schaefer 100 → Desikan-Killiany 좌표 매핑

## 왜 이 작업이 필요한가

선행연구(Nature Mental Health 2024, "Discriminative functional connectivity signature of
cocaine use disorder links to rTMS treatment response")는 우리와 **같은 SUDMEX-TMS 코호트**
(active 25 / sham 20)에서 baseline FC로 rTMS 반응(ΔVAS)을 예측했다.

그 논문이 쓴 파셀레이션은 **Schaefer 100 (7Networks)** 이고,
우리 FC 데이터는 **Desikan-Killiany 68 cortical + FreeSurfer aseg 16 subcortical = 84** 이다.
두 아틀라스는 분할 기준이 달라(기능적 경계 vs 해부학적 고랑) 파셀 간 1:1 대응이 원리적으로 없다.

따라서 논문이 지목한 영역을 우리 아틀라스에서 **근사**해야 하고,
그 근사를 추론이 아니라 **좌표 계산**으로 수행하는 것이 이 노트북의 목적이다.

## 논문에서 가져올 수 있는 것 / 없는 것

**없는 것** — 논문의 진단용 FC 마스크(수백 개 엣지)는 좌표나 표로 공개되지 않았다.
Figure S1에 그림으로만 있어 목록을 읽어낼 수 없다. 상위 40개 hyper/hypo connection도
"for better visualization" 목적의 그림이며 텍스트 목록이 없다.

**있는 것** — 이름이 명시된 엣지 2개 (반응 예측용, Figure S9A/B):

| 엣지 | 한쪽 | 다른쪽 |
|---|---|---|
| 1 | LH LIM OFC 1 | RH Cont PFCmp 2 |
| 2 | LH DMN Temp 3 | RH LIM OFC 1 |

이 4개 파셀만이 재현 가능한 출발점이다.

## 파셀 이름 이력 문제

논문이 쓴 이름 중 2개는 **현행 Schaefer centroid 파일에 존재하지 않는다.**
CBIG가 2019-09-16에 라벨 매칭 알고리즘 버그를 고치면서 이름을 바꿨기 때문이다
(파셀 경계는 그대로, 라벨명만 수정). 100Parcels_7Networks에서 16개 component name,
3개 component number가 변경되었다.

변경 로그(`Update_20190916_*.csv`)로 대조한 결과:

| 논문 라벨 (구버전) | 파셀 인덱스 | 현행 이름 |
|---|---|---|
| LH Limbic OFC 1 | 31 | `7Networks_LH_Limbic_OFC_1` (변경 없음) |
| RH Cont PFCmp 2 | 88 | `7Networks_RH_Cont_PFCmp_1` |
| LH Default Temp 3 | 40 | `7Networks_LH_Default_Par_1` |
| RH Limbic OFC 1 | 79 | `7Networks_RH_Limbic_OFC_1` (변경 없음) |

**주목** — 파셀 40번은 `Temp`(측두) → `Par`(두정)로 바뀌었다. 즉 구버전 이름이 틀렸던 것이다.
논문은 이를 "좌 중측두피질(middle temporal cortex)"로 해석했으나, 실제 중심좌표는
(-58, -50, 12)로 측두-두정 접합부에 해당한다. 논문의 해부학적 서술에 오류가 있다.

## 매핑 방법과 한계

Schaefer 파셀의 MNI 중심좌표(CBIG 공개)와 DK 영역의 MNI 중심좌표(brainGraph R 패키지,
RRID: SCR_017260)를 같은 반구 안에서 유클리드 거리로 비교해 최근접 영역을 찾는다.

**한계 세 가지 — 논문 Methods/Limitations에 반드시 명시할 것:**

1. 중심점 하나로 대응시키므로 파셀 전체의 중첩을 반영하지 못한다. 정확히 하려면
   두 아틀라스의 MNI 볼륨을 복셀 단위로 겹쳐 중첩률을 계산해야 한다.
2. DK 영역은 Schaefer 파셀보다 훨씬 크다(`superiorfrontal` 등). 큰 영역은 중심점이
   실제 범위를 대표하지 못한다.
3. brainGraph의 DK 좌표는 패키지 저자가 fsaverage 라벨 평균을 기준으로 하되
   일부를 MNI152 볼륨에서 육안 확인 후 수동 조정한 값이다. 공식 표준 좌표가 아니다.

**판정 기준** — 1순위 거리가 20mm를 넘거나, 1순위와 2순위 거리가 비슷하면
그 매핑은 신뢰할 수 없다고 보고 논문에 그대로 기술한다.

---
## 0. 준비

필요 파일 (모두 같은 폴더에 둘 것):
- `Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm_Centroid_RAS.csv` — CBIG GitHub
- `dk.rda` — brainGraph CRAN tarball의 `data/dk.rda`
- `Update_20190916_component_name_changes.csv` / `..._number_changes.csv` — CBIG Updates 폴더 (검증용)

In [8]:
# pyreadr: R의 .rda 파일을 pandas DataFrame으로 읽기 위해 필요
!pip install pyreadr

import numpy as np
import pandas as pd
import pyreadr

SCHAEFER_CSV = 'Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.Centroid_RAS.csv'
DK_RDA       = 'dk.rda'
RENAME_NAME  = 'Update_20190916_component_name_changes.csv'
RENAME_NUM   = 'Update_20190916_component_number_changes.csv'

---
## 1. 논문 라벨 → 현행 파셀 인덱스 확인

논문이 쓴 구버전 이름을 CBIG 변경 로그에서 찾아 인덱스를 확정한다.
추론이 아니라 로그 대조 결과임을 남기기 위해 이 셀을 별도로 둔다.

In [9]:
renames = pd.concat([pd.read_csv(RENAME_NAME), pd.read_csv(RENAME_NUM)])
renames = renames[renames['Resolution'] == '100Parcels_7Networks']

# 논문(Figure S9A, S9B)이 이름을 명시한 4개 파셀 — 구버전 표기
PAPER_LABELS = {
    'edge1_a': '7Networks_LH_Limbic_OFC_1',   # LH LIM OFC 1
    'edge1_b': '7Networks_RH_Cont_PFCmp_2',   # RH Cont PFCmp 2
    'edge2_a': '7Networks_LH_Default_Temp_3', # LH DMN Temp 3
    'edge2_b': '7Networks_RH_Limbic_OFC_1',   # RH LIM OFC 1
}

print('--- CBIG 2019-09-16 변경 로그에서 해당 파셀 조회 ---')
hit = renames[renames['Old parcel name'].isin(PAPER_LABELS.values())]
print(hit.to_string(index=False) if len(hit) else '(변경 이력 없음)')

--- CBIG 2019-09-16 변경 로그에서 해당 파셀 조회 ---
          Resolution  Old parcel index             Old parcel name  New parcel index            New parcel name
100Parcels_7Networks                40 7Networks_LH_Default_Temp_3                40 7Networks_LH_Default_Par_1
100Parcels_7Networks                88   7Networks_RH_Cont_PFCmp_2                88  7Networks_RH_Cont_PFCmp_1


In [10]:
# 변경 로그 대조로 확정한 인덱스
TARGETS = {
    31: 'LH LIM OFC 1',      # 이름 변경 없음
    88: 'RH Cont PFCmp 2',   # -> RH_Cont_PFCmp_1 (component number 변경)
    40: 'LH DMN Temp 3',     # -> LH_Default_Par_1 (component name 변경, Temp가 오표기였음)
    79: 'RH LIM OFC 1',      # 이름 변경 없음
}

sch = pd.read_csv(SCHAEFER_CSV)
print(sch[sch['ROI Label'].isin(TARGETS)].to_string(index=False))

 ROI Label                   ROI Name   R   A   S
        31  7Networks_LH_Limbic_OFC_1 -14  32 -20
        40 7Networks_LH_Default_Par_1 -58 -50  12
        79  7Networks_RH_Limbic_OFC_1  12  34 -20
        88  7Networks_RH_Cont_PFCmp_1   6  28  30


---
## 2. DK 좌표표 로드

brainGraph의 `dk` 데이터프레임: 68행(피질만), 컬럼은 `name`, `x.mni`, `y.mni`, `z.mni`,
`lobe`, `hemi`, `index`, `name.full`.

Schaefer는 피질 전용 아틀라스이고 매핑 대상 4개도 모두 피질이므로 `dk`(68)로 충분하다
(`dk.scgm` 82개 버전은 불필요).

In [11]:
dk = pyreadr.read_r(DK_RDA)['dk']
print(dk.shape)
print(dk.head().to_string())

(68, 8)
    name  x.mni  y.mni  z.mni       lobe hemi  index                               name.full
0  lBSTS  -56.0  -44.0    5.0   Temporal    L      1  L bank of the superior temporal sulcus
1  lcACC   -2.0   21.0   27.0  Cingulate    L      2             L caudal anterior cingulate
2  lcMFG  -45.0   18.0   46.0    Frontal    L      3           L caudal middle frontal gyrus
3   lCUN   -1.0  -82.0   20.0  Occipital    L      4                                L cuneus
4   lENT  -16.0  -10.0  -29.0   Temporal    L      5                            L entorhinal


---
## 3. 최근접 DK 영역 탐색

같은 반구 안에서만 비교한다. 1순위만 보면 위험하므로 상위 3개와 거리를 함께 출력해
매핑의 신뢰도를 직접 판단할 수 있게 한다.

In [12]:
def nearest_dk(parcel_idx, top_n=3):
    row = sch[sch['ROI Label'] == parcel_idx].iloc[0]
    coord = np.array([row['R'], row['A'], row['S']], dtype=float)
    hemi = 'L' if '_LH_' in row['ROI Name'] else 'R'

    cand = dk[dk['hemi'] == hemi].copy()
    cand['dist_mm'] = np.sqrt(
        ((cand[['x.mni', 'y.mni', 'z.mni']].values - coord) ** 2).sum(axis=1))
    return row, coord, cand.nsmallest(top_n, 'dist_mm')


records = []
for idx, paper_label in TARGETS.items():
    row, coord, top = nearest_dk(idx)
    print(f"\n{'='*72}")
    print(f"논문 라벨 : {paper_label}")
    print(f"현행 이름 : {row['ROI Name']}  (index {idx})")
    print(f"MNI 좌표  : {coord}")
    print(top[['name', 'name.full', 'x.mni', 'y.mni', 'z.mni', 'dist_mm']]
          .round(1).to_string(index=False))

    best = top.iloc[0]
    gap = top.iloc[1]['dist_mm'] - best['dist_mm']
    flags = []
    if best['dist_mm'] > 20:
        flags.append('거리 20mm 초과')
    if gap < 5:
        flags.append(f'1-2순위 차이 {gap:.1f}mm로 모호')
    print('판정 :', ' / '.join(flags) if flags else '단일 후보로 명확')

    records.append({
        'paper_label': paper_label,
        'schaefer_current': row['ROI Name'],
        'parcel_idx': idx,
        'mni': tuple(coord),
        'dk_best': best['name.full'],
        'dist_mm': round(best['dist_mm'], 1),
        'runner_up': top.iloc[1]['name.full'],
        'gap_mm': round(gap, 1),
    })


논문 라벨 : LH LIM OFC 1
현행 이름 : 7Networks_LH_Limbic_OFC_1  (index 31)
MNI 좌표  : [-14.  32. -20.]
name               name.full  x.mni  y.mni  z.mni  dist_mm
lMOF  L medial orbitofrontal   -4.0   44.0  -14.0     16.7
lLOF L lateral orbitofrontal  -36.0   30.7  -12.1     23.4
 lTP         L temporal pole  -26.0   15.0  -35.0     25.7
판정 : 단일 후보로 명확

논문 라벨 : RH Cont PFCmp 2
현행 이름 : 7Networks_RH_Cont_PFCmp_1  (index 88)
MNI 좌표  : [ 6. 28. 30.]
 name                           name.full  x.mni  y.mni  z.mni  dist_mm
rcACC         R caudal anterior cingulate    3.0   21.0   27.0      8.2
 rSFG            R superior frontal gyrus   16.0   34.0   53.0     25.8
rrACC R rostral anterior cingulate cortex    4.0   38.0    4.0     27.9
판정 : 단일 후보로 명확

논문 라벨 : LH DMN Temp 3
현행 이름 : 7Networks_LH_Default_Par_1  (index 40)
MNI 좌표  : [-58. -50.  12.]
 name                              name.full  x.mni  y.mni  z.mni  dist_mm
lBSTS L bank of the superior temporal sulcus  -56.0  -44.0    5.0      9.4
lSMAR    

In [13]:
mapping = pd.DataFrame(records)
print(mapping.to_string(index=False))
# mapping.to_csv('schaefer_to_dk_mapping.csv', index=False)

    paper_label           schaefer_current  parcel_idx                  mni                                dk_best  dist_mm                runner_up  gap_mm
   LH LIM OFC 1  7Networks_LH_Limbic_OFC_1          31 (-14.0, 32.0, -20.0)                 L medial orbitofrontal     16.7  L lateral orbitofrontal     6.7
RH Cont PFCmp 2  7Networks_RH_Cont_PFCmp_1          88    (6.0, 28.0, 30.0)            R caudal anterior cingulate      8.2 R superior frontal gyrus    17.6
  LH DMN Temp 3 7Networks_LH_Default_Par_1          40 (-58.0, -50.0, 12.0) L bank of the superior temporal sulcus      9.4    L supramarginal gyrus    19.1
   RH LIM OFC 1  7Networks_RH_Limbic_OFC_1          79  (12.0, 34.0, -20.0)                R lateral orbitofrontal      9.0   R medial orbitofrontal     5.0


---
## 4. 결과 해석 시 유의사항

**이 매핑으로 얻는 것은 엣지 2개뿐이다.** 논문의 마스크 전체(수백 개)는 재현 불가이므로,
"논문 방법 재현"이 아니라 "논문이 보고한 영역을 우리 아틀라스에서 근사한 탐색적 분석"으로
포지셔닝해야 한다.

**다음 단계 선택지:**

- (a) 매핑된 DK 영역들 사이의 **엣지**만 골라 씀 — 논문의 특정 쌍 구조를 보존
- (b) 매핑된 DK 영역의 **node strength**만 씀 — 다루기 쉬우나 "어디와 연결됐나" 정보 손실

**표본 관련 주의:**

- FC 데이터는 n=44 (sub-001은 ses-t0 fmriprep 크래시로 MNI preproc BOLD 없음)
- morph 결과(n=45)와 직접 비교하려면 morph도 rid=1 제외 후 재실행해야 표본이 맞음
- FC의 교란변수는 age, sex, **mean_fd** (morph의 eTIV 자리를 대체)

**아틀라스 표기 주의:**

FC 파일명이 `FreeSurferDKT_`이지만 실제 라벨 세트는 **DK**다.
(`bankssts`, `frontalpole`, `temporalpole`이 포함되어 있으며 이는 DKT에서 제거된 라벨이다.
DKT는 62개, DK는 68개.) 논문에는 다음과 같이 표기할 것:

> Desikan-Killiany atlas (68 cortical regions) and FreeSurfer aseg (16 subcortical structures)